In [1]:
import pandas as pd 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [2]:
train_txn  = pd.read_csv('../data/raw/train_transaction.csv')
train_id   = pd.read_csv('../data/raw/train_identity.csv')
test_txn   = pd.read_csv('../data/raw/test_transaction.csv')
test_id    = pd.read_csv('../data/raw/test_identity.csv')

# Merge on TransactionID — left join keeps all transactions
train = train_txn.merge(train_id, on='TransactionID', how='left')
test  = test_txn.merge(test_id,  on='TransactionID', how='left')

In [3]:
import pandas as pd
import numpy as np

def reduce_mem_usage(df, verbose=True):
    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        if col_type == object:
            if df[col].nunique() / len(df[col]) < 0.5:
                df[col] = df[col].astype('category')

        elif col_type == bool:
            df[col] = df[col].astype(np.int8)

        elif pd.api.types.is_integer_dtype(col_type):
            # Skip if column has nulls — int types can't hold NaN
            if df[col].isnull().any():
                continue
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo(np.int8).min  and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)

        elif pd.api.types.is_float_dtype(col_type):
            # Drop non-finite values from the min/max check — they break finfo comparisons
            finite_vals = df[col].dropna()
            finite_vals = finite_vals[np.isfinite(finite_vals)]
            if finite_vals.empty:
                continue
            c_min = float(finite_vals.min())
            c_max = float(finite_vals.max())
            # float16 is very narrow — skip it, causes more problems than it saves
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f"Memory: {start_mem:.1f} MB → {end_mem:.1f} MB "
              f"({100*(start_mem-end_mem)/start_mem:.1f}% reduction)")
    return df

train = reduce_mem_usage(train)
test  = reduce_mem_usage(test)

Memory: 2514.0 MB → 1603.3 MB (36.2% reduction)
Memory: 2164.1 MB → 1386.1 MB (35.9% reduction)


In [4]:
# Drop columns missing more than 90% of values
miss_pct = train.isnull().mean()
drop_cols = miss_pct[miss_pct > 0.90].index.tolist()
print(f"Dropping {len(drop_cols)} columns with >90% missing: {drop_cols[:10]}...")

train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, errors='ignore', inplace=True)

# Drop near-zero variance columns (they carry no signal anyway)
from sklearn.feature_selection import VarianceThreshold
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != 'isFraud']

selector = VarianceThreshold(threshold=0.01)
selector.fit(train[num_cols].fillna(0))
low_var = [c for c, s in zip(num_cols, selector.get_support()) if not s]
print(f"Dropping {len(low_var)} near-zero variance columns")
train.drop(columns=low_var, inplace=True)
test.drop(columns=low_var, errors='ignore', inplace=True)

Dropping 12 columns with >90% missing: ['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25']...
Dropping 18 near-zero variance columns


In [5]:
import gc
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

v_cols = [c for c in train.columns if c.startswith('V')]

miss_patterns = train[v_cols].isnull().T.apply(
    lambda r: r.values.tobytes(), axis=1
)
groups = {}
for col, pat in miss_patterns.items():
    groups.setdefault(pat, []).append(col)

print(f"V-feature groups: {len(groups)}, sizes: {[len(g) for g in groups.values()]}")

pca_results_tr, pca_results_te = [], []

for i, group_cols in enumerate(groups.values()):
    if len(group_cols) < 2:
        continue

    n_components = min(6, len(group_cols))
    imp = SimpleImputer(strategy='median')
    pca = PCA(n_components=n_components, random_state=42)

    tr_imp = imp.fit_transform(train[group_cols])
    te_imp = imp.transform(test[group_cols])

    tr_pca = pca.fit_transform(tr_imp)
    te_pca = pca.transform(te_imp)

    explained = pca.explained_variance_ratio_.sum()
    print(f"Group {i+1} ({len(group_cols)} cols) → {n_components} PCs, "
          f"variance explained: {explained:.1%}")

    cols = [f'V_pca_g{i}_pc{j}' for j in range(n_components)]
    pca_results_tr.append(pd.DataFrame(tr_pca, columns=cols, index=train.index))
    pca_results_te.append(pd.DataFrame(te_pca, columns=cols, index=test.index))

train = pd.concat([train.drop(columns=v_cols)] + pca_results_tr, axis=1)
test  = pd.concat([test.drop(columns=v_cols)]  + pca_results_te, axis=1)
gc.collect()

print(f"Shape after V-PCA: {train.shape}")
print(f"Memory: {train.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

V-feature groups: 15, sizes: [11, 21, 18, 21, 19, 31, 17, 11, 31, 19, 46, 16, 31, 11, 18]
Group 1 (11 cols) → 6 PCs, variance explained: 98.1%
Group 2 (21 cols) → 6 PCs, variance explained: 94.1%
Group 3 (18 cols) → 6 PCs, variance explained: 94.2%
Group 4 (21 cols) → 6 PCs, variance explained: 92.4%
Group 5 (19 cols) → 6 PCs, variance explained: 92.1%
Group 6 (31 cols) → 6 PCs, variance explained: 100.0%
Group 7 (17 cols) → 6 PCs, variance explained: 100.0%
Group 8 (11 cols) → 6 PCs, variance explained: 100.0%
Group 9 (31 cols) → 6 PCs, variance explained: 100.0%
Group 10 (19 cols) → 6 PCs, variance explained: 100.0%
Group 11 (46 cols) → 6 PCs, variance explained: 99.7%
Group 12 (16 cols) → 6 PCs, variance explained: 100.0%
Group 13 (31 cols) → 6 PCs, variance explained: 99.9%
Group 14 (11 cols) → 6 PCs, variance explained: 100.0%
Group 15 (18 cols) → 6 PCs, variance explained: 100.0%
Shape after V-PCA: (590540, 173)
Memory: 983.5 MB


In [6]:
def feature_engineering(df):
    # ── Log-transform skewed amount ──────────────────────────────
    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
    df['TransactionAmt_decimal'] = df['TransactionAmt'] - np.floor(df['TransactionAmt'])
    # Round amounts (99.00) are less suspicious than odd ones (99.47)

    # ── Time features ────────────────────────────────────────────
    df['hour']     = (df['TransactionDT'] // 3600) % 24
    df['day']      = (df['TransactionDT'] // 86400) % 7
    df['week']     = df['TransactionDT'] // 604800

    # ── Pseudo-user ID (your biggest EDA insight) ────────────────
    # Card1 alone isn't a user. Card1 + addr1 + email together is.
    df['uid'] = (df['card1'].astype(str) + '_' +
                 df['addr1'].astype(str) + '_' +
                 df['P_emaildomain'].astype(str))

    # ── Frequency encoding (replaces high-cardinality cats) ──────
    for col in ['card1','card2','card3','card5','uid',
                'P_emaildomain','R_emaildomain','DeviceInfo']:
        freq = df[col].value_counts()
        df[f'{col}_freq'] = df[col].map(freq)

    # ── Aggregation features (card-level transaction history) ────
    for col in ['TransactionAmt', 'TransactionAmt_log', 'D1', 'D15']:
        grp = df.groupby('card1')[col]
        df[f'card1_{col}_mean'] = df['card1'].map(grp.mean())
        df[f'card1_{col}_std']  = df['card1'].map(grp.std())

    # Ratio of this transaction to card's historical average
    df['amt_vs_card_mean'] = (df['TransactionAmt'] /
                               (df['card1_TransactionAmt_mean'] + 1))

    # ── Email domain match (purchaser vs recipient) ───────────────
    df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)

    # ── Missingness flags for D-features ─────────────────────────
    d_cols = [c for c in df.columns if c.startswith('D')]
    for col in d_cols:
        df[f'{col}_isna'] = df[col].isna().astype(int)

    # ── Identity presence flag (from your EDA) ───────────────────
    id_cols = [c for c in df.columns if c.startswith('id')]
    df['has_identity'] = df[id_cols].notnull().any(axis=1).astype(int)
    df['identity_richness'] = df[id_cols].notnull().sum(axis=1)

    return df

train = feature_engineering(train)
test  = feature_engineering(test)

In [7]:
from sklearn.preprocessing import LabelEncoder

# Time-based split
split_time = train['TransactionDT'].quantile(0.8)
tr = train[train['TransactionDT'] <= split_time].copy()
va = train[train['TransactionDT'] >  split_time].copy()

# Only encode cols that exist in BOTH train and test
cat_cols = [c for c in train.select_dtypes('object').columns.tolist()
            if c in test.columns]

# Also catch 'category' dtype columns (from reduce_mem_usage)
cat_cols += [c for c in train.select_dtypes('category').columns.tolist()
             if c in test.columns and c not in cat_cols]

print(f"Encoding {len(cat_cols)} categorical columns: {cat_cols}")

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    tr[col]   = le.transform(tr[col].astype(str))
    va[col]   = le.transform(va[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# Columns that exist only in train — just drop them, test can't use them
train_only_cats = [c for c in train.select_dtypes(['object','category']).columns
                   if c not in test.columns]
if train_only_cats:
    print(f"Dropping {len(train_only_cats)} cols missing from test: {train_only_cats}")
    tr.drop(columns=train_only_cats, inplace=True)
    va.drop(columns=train_only_cats, inplace=True)

# Define feature set
DROP_COLS = ['isFraud', 'TransactionID', 'TransactionDT']
FEATURES  = [c for c in tr.columns if c not in DROP_COLS and c in test.columns]

X_tr, y_tr = tr[FEATURES], tr['isFraud']
X_va, y_va = va[FEATURES], va['isFraud']
X_te       = test[FEATURES]

print(f"\nFeature count : {len(FEATURES)}")
print(f"Train fold    : {X_tr.shape}, fraud {y_tr.mean():.3%}")
print(f"Val fold      : {X_va.shape}, fraud {y_va.mean():.3%}")

Encoding 17 categorical columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'DeviceType', 'DeviceInfo', 'uid']
Dropping 13 cols missing from test: ['id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38']

Feature count : 185
Train fold    : (472432, 185), fraud 3.514%
Val fold      : (118108, 185), fraud 3.441%


In [8]:
# Stage 3 — Baseline + model training
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# ── Logistic regression baseline ─────────────────────────────────
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

lr_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])
lr_pipe.fit(X_tr, y_tr)
lr_prob = lr_pipe.predict_proba(X_va)[:, 1]
print(f"LR  AUC-ROC : {roc_auc_score(y_va, lr_prob):.4f}")
print(f"LR  PR-AUC  : {average_precision_score(y_va, lr_prob):.4f}")

# ── LightGBM ──────────────────────────────────────────────────────
lgb_params = {
    'objective'        : 'binary',
    'metric'           : 'auc',
    'boosting_type'    : 'gbdt',
    'n_estimators'     : 2000,
    'learning_rate'    : 0.05,
    'num_leaves'       : 256,
    'max_depth'        : -1,
    'min_child_samples': 20,
    'feature_fraction' : 0.8,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 5,
    'is_unbalance'     : True,   # handles 1:27 imbalance
    'random_state'     : 42,
    'verbose'          : -1,
}

lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
)

lgb_prob = lgb_model.predict_proba(X_va)[:, 1]
print(f"\nLGB AUC-ROC : {roc_auc_score(y_va, lgb_prob):.4f}")
print(f"LGB PR-AUC  : {average_precision_score(y_va, lgb_prob):.4f}")

LR  AUC-ROC : 0.7875
LR  PR-AUC  : 0.1364
Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.926411
Early stopping, best iteration is:
[222]	valid_0's auc: 0.927322

LGB AUC-ROC : 0.9273
LGB PR-AUC  : 0.5838


In [9]:
# Stage 4 — Hyperparameter tuning with Optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'objective'        : 'binary',
        'metric'           : 'auc',
        'verbosity'        : -1,
        'boosting_type'    : 'gbdt',
        'is_unbalance'     : True,
        'n_estimators'     : 3000,
        'learning_rate'    : trial.suggest_float('lr', 0.01, 0.1, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 64, 512),
        'max_depth'        : trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'feature_fraction' : trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction' : trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq'     : trial.suggest_int('bagging_freq', 1, 10),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state'     : 42,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(-1)])
    prob = model.predict_proba(X_va)[:, 1]
    return roc_auc_score(y_va, prob)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest AUC-ROC : {study.best_value:.4f}")
print(f"Best params  : {study.best_params}")

ModuleNotFoundError: No module named 'optuna'

In [ ]:
# Stage 5 — Proper model evaluation
from sklearn.metrics import (roc_curve, precision_recall_curve,
                              confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── ROC curve ────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_va, lgb_prob)
auc_roc = roc_auc_score(y_va, lgb_prob)
axes[0].plot(fpr, tpr, color='#2980b9', lw=2, label=f'LGB (AUC={auc_roc:.4f})')
axes[0].plot([0,1],[0,1],'--', color='gray', label='Random (0.5000)')
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC curve')
axes[0].legend()

# ── Precision-Recall curve ────────────────────────────────────────
prec, rec, thresholds = precision_recall_curve(y_va, lgb_prob)
pr_auc = average_precision_score(y_va, lgb_prob)
axes[1].plot(rec, prec, color='#e74c3c', lw=2, label=f'LGB (PR-AUC={pr_auc:.4f})')
axes[1].axhline(y_va.mean(), color='gray', linestyle='--',
                label=f'Baseline ({y_va.mean():.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall curve')
axes[1].legend()

# ── Threshold tuning ──────────────────────────────────────────────
# Find threshold that maximises F2 score (recall-weighted — missing fraud is costly)
from sklearn.metrics import fbeta_score
f2_scores = [fbeta_score(y_va, lgb_prob >= t, beta=2)
             for t in thresholds]
best_thresh = thresholds[np.argmax(f2_scores)]
print(f"\nBest F2 threshold : {best_thresh:.4f}")

# Confusion matrix at best threshold
y_pred = (lgb_prob >= best_thresh).astype(int)
cm = confusion_matrix(y_va, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Legit','Fraud']).plot(ax=axes[2], cmap='Blues')
axes[2].set_title(f'Confusion matrix @ threshold={best_thresh:.3f}')

plt.tight_layout()
plt.savefig('05_model_evaluation.png', dpi=150)
plt.show()

In [ ]:
# Stage 7 — Ensemble & final predictions
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=27.6,   # imbalance ratio
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    early_stopping_rounds=100,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=200)
xgb_prob = xgb_model.predict_proba(X_va)[:, 1]

# ── Rank averaging (more robust than simple average) ─────────────
from scipy.stats import rankdata

def rank_avg(*probs):
    ranks = [rankdata(p) / len(p) for p in probs]
    return np.mean(ranks, axis=0)

ensemble_prob = rank_avg(lgb_prob, xgb_prob)
print(f"\nEnsemble AUC-ROC : {roc_auc_score(y_va, ensemble_prob):.4f}")
print(f"Ensemble PR-AUC  : {average_precision_score(y_va, ensemble_prob):.4f}")

# ── Generate submission ───────────────────────────────────────────
lgb_test  = lgb_model.predict_proba(X_te)[:, 1]
xgb_test  = xgb_model.predict_proba(X_te)[:, 1]
test_prob = rank_avg(lgb_test, xgb_test)

submission = pd.DataFrame({
    'TransactionID' : test['TransactionID'],
    'isFraud'       : test_prob
})
submission.to_csv('submission.csv', index=False)
print(submission.head())

In [10]:
# !pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------------------------ --------------- 1.3/2.1 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 7.2 MB/s  0:00:00

   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------------- -------------------------- 2/6 [colorlog]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   ---------